### Instalación de requerimientos e importación de bibliotecas

In [1]:
%pip install -r requirements.txt
import os
import gdown
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import contextily as ctx
import folium
import re

import polars as pl
from IPython.display import display


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Función para descarga de datasets con gdown


In [2]:
# URL de Google Drive
url = "https://drive.google.com/uc?id=ID_DEL_ARCHIVO"

def descargar_archivos_desde_drive(lista_ids):
    """
    Descarga varios archivos de Google Drive a partir de una lista de IDs.

    Args:
        lista_ids (list): Una lista de strings con los IDs de los archivos de Drive.
    """
    for file_id in lista_ids:
        # Construimos la URL de descarga directa
        url = f'https://drive.google.com/uc?id={file_id}'

        print(f"\nIniciando descarga del ID: {file_id}")

        try:
            # gdown detecta automáticamente el nombre original del archivo
            gdown.download(url, quiet=False)
        except Exception as e:
            print(f"No se pudo descargar el archivo {file_id}. Error: {e}")

In [3]:
# Definir la carpeta y el archivo de destino
output_dir = "./content/csv"

# Crear la carpeta si no existe
os.makedirs(output_dir, exist_ok=True)

# Se cambia a la carpeta de descaga de CSV
os.chdir(output_dir)

### Carga de CSV desde Google Drive

In [4]:
'''

lista_ids_datasets = [
    '1sWI3jP6f9VDJ-IE1EI1lkc8rkxooP3b7', # lineas-de-subte
    '1V6Cjhf2QU_gcig6HT5EvXhk0n2Egerqr', # estaciones-accesibles
    '1CyWPBgfAYRBcYvQO7U7cRlAoQbGWoosP', # historico_2014
    '1g7LpNJFqNcqDmgaGc81MMgV3VfrD6Ah6', # historico_2015
    '1QqOb3oLoMs014d2YBYw4_e8ww4jo-JX1', # historico_2016
    '1G7noINplTWyt2g9xRrS7l0BKgFOW05hv', # historico_2017
    '11WgJxZsC4zUURSlCUBEQKXCQK5RLkRNZ', # historico_2018
    '1DVgvubSgYCTPQCfA4zj5eiH_ni136o9A', # historico_2019
    '1hlfAVsJS20InzvIkUXT4t5_nOgd2m1NW', # historico_2020
    '1hW4qHioTzrXlDnBfWextpMQpar0kmL03', # historico_2021
    '1yETNbct23DLqYoN7ti6hNV3RdKErwMYI', # registro-historico-del-precio-del-boleto
    '1mY28zAPaI79Pt-OLoNSAhIxnCtflpmKU', # registro-historico-del-precio-del-boleto.xlsx
    '1PEAW6Vik2k-J7k9C6gxNtl_2df636Q41'  # viajes_anual
    ]

descargar_archivos_desde_drive(lista_ids_datasets)
'''

"\n\nlista_ids_datasets = [\n    '1sWI3jP6f9VDJ-IE1EI1lkc8rkxooP3b7', # lineas-de-subte\n    '1V6Cjhf2QU_gcig6HT5EvXhk0n2Egerqr', # estaciones-accesibles\n    '1CyWPBgfAYRBcYvQO7U7cRlAoQbGWoosP', # historico_2014\n    '1g7LpNJFqNcqDmgaGc81MMgV3VfrD6Ah6', # historico_2015\n    '1QqOb3oLoMs014d2YBYw4_e8ww4jo-JX1', # historico_2016\n    '1G7noINplTWyt2g9xRrS7l0BKgFOW05hv', # historico_2017\n    '11WgJxZsC4zUURSlCUBEQKXCQK5RLkRNZ', # historico_2018\n    '1DVgvubSgYCTPQCfA4zj5eiH_ni136o9A', # historico_2019\n    '1hlfAVsJS20InzvIkUXT4t5_nOgd2m1NW', # historico_2020\n    '1hW4qHioTzrXlDnBfWextpMQpar0kmL03', # historico_2021\n    '1yETNbct23DLqYoN7ti6hNV3RdKErwMYI', # registro-historico-del-precio-del-boleto\n    '1mY28zAPaI79Pt-OLoNSAhIxnCtflpmKU', # registro-historico-del-precio-del-boleto.xlsx\n    '1PEAW6Vik2k-J7k9C6gxNtl_2df636Q41'  # viajes_anual\n    ]\n\ndescargar_archivos_desde_drive(lista_ids_datasets)\n"

### Carga de CSV en dataframes de Polars (columnar)

In [5]:
null_values = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]

for anio in range(2014, 2022):
    ruta_csv = f"./historico_{anio}.csv" 
    
    try:
        # Leemos solo 5 filas para extraer el esquema de datos (Schema)
        df_esquema = pl.read_csv(ruta_csv, n_rows=5, null_values=null_values)
        
        print(f"ESQUEMA DE COLUMNAS - AÑO {anio}")
        print(f"• Columnas detectadas: {df_esquema.columns}")
        # Te muestra el par 'nombre_columna': TipoDeDato
        print(f"• Tipos de datos: {df_esquema.schema}\n")
        print("-" * 70)
        
    except FileNotFoundError:
        print(f"No se encontró el archivo para el año {anio} en {ruta_csv}.")

ESQUEMA DE COLUMNAS - AÑO 2014
• Columnas detectadas: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ID_ESTACION', 'ESTACION', 'PAX_PAGO', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
• Tipos de datos: Schema({'FECHA': String, 'DESDE': String, 'HASTA': String, 'LINEA': String, 'MOLINETE': String, 'ID_ESTACION': Int64, 'ESTACION': String, 'PAX_PAGO': Int64, 'PAX_PASES_PAGOS': Int64, 'PAX_FRANQ': Int64, 'PAX_TOTAL': Int64})

----------------------------------------------------------------------
ESQUEMA DE COLUMNAS - AÑO 2015
• Columnas detectadas: ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
• Tipos de datos: Schema({'periodo': Int64, 'fecha': String, 'desde': String, 'hasta': String, 'linea': String, 'molinete': String, 'estacion': String, 'pax_pagos': Float64, 'pax_pases_pagos': Float64, 'pax_franq': Float64, 'total': Float64})

----------------------------------------------------------------------
ESQU

Año 2014 se detecta PAX_PAGO en lugar de PAX_PAGOS como el resto de los datasets.

Año 2021 detecta una sola columna por estar separado por punto y coma en lugar de coma

In [6]:
COLUMNAS_DESEADAS = [
    "FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION", "PAX_PAGO",
    "PAX_PAGOS", "PAX_PASES_PAGOS", 
    "PAX_FREQ", "PAX_FRANQ", "PAX_TOTAL", "TOTAL"
]

valores_nulos = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]

for anio in range(2014, 2022):
    ruta_csv = f"historico_{anio}.csv"
    
    try:
        separador_actual = ";" if anio == 2021 else ","
        
        # Si es 2016 o 2017, aplicamos la medicina fuerte de entrada
        if anio in [2016, 2017]:
            lf_temp = pl.scan_csv(
                ruta_csv, 
                null_values=valores_nulos, 
                infer_schema_length=0,
                separator=separador_actual,
                # Forzamos pérdida tolerable de caracteres rotos para que NO explote
                encoding="utf8-lossy" 
            )
            encoding_usado = "UTF-8 (Lossy - Caracteres rebeldes omitidos)"
        else:
            lf_temp = pl.scan_csv(
                ruta_csv, 
                null_values=valores_nulos, 
                infer_schema_length=0,
                separator=separador_actual,
                encoding="utf8"
            )
            encoding_usado = "UTF-8 Estándar"
        
        # Extraemos las columnas reales
        columnas_reales = lf_temp.collect_schema().names()
        
        # Filtramos las columnas
        columnas_a_importar = [
            col for col in columnas_reales 
            if col.upper() in COLUMNAS_DESEADAS
        ]
        
        # Traemos a memoria
        df_año = lf_temp.select(columnas_a_importar).collect()
        
        # Guardamos en el entorno global
        globals()[f"historico_{anio}_df"] = df_año
        
        print(f"{ruta_csv} cargado con éxito.")
        print(f"   • Separador: '{separador_actual}' | Codificación: {encoding_usado}")
        print(f"   • Columnas importadas: {columnas_a_importar}")
        print(f"   • Dimensiones: {df_año.shape}\n")
        print("-" * 75)
        
    except FileNotFoundError:
        print(f"No se encontró el archivo: {ruta_csv}")
    except Exception as e:
        print(f"Error crítico insalvable en el año {anio}: {e}")

historico_2014.csv cargado con éxito.
   • Separador: ',' | Codificación: UTF-8 Estándar
   • Columnas importadas: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGO', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
   • Dimensiones: (10857244, 10)

---------------------------------------------------------------------------
historico_2015.csv cargado con éxito.
   • Separador: ',' | Codificación: UTF-8 Estándar
   • Columnas importadas: ['fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
   • Dimensiones: (10958582, 10)

---------------------------------------------------------------------------
historico_2016.csv cargado con éxito.
   • Separador: ',' | Codificación: UTF-8 (Lossy - Caracteres rebeldes omitidos)
   • Columnas importadas: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FREQ', 'TOTAL']
   • Dimensiones: (11542322, 10)

------------------------

In [7]:
print("==================================================================")
print("REESTRUCTURACIÓN DE NOMENCLATURAS PARA EL DATASET")
print("==================================================================")

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        columnas_actuales = df_actual.columns
        
        # Armamos un diccionario dinámico de cambios para este año en particular
        diccionario_renombrado = {}
        
        # 1. Pasamos PAX_FREQ (o variantes) a PAX_FRANQ
        for col in columnas_actuales:
            if col.upper() in ["PAX_FREQ", "PAX_FRANQUICIAS"]:
                diccionario_renombrado[col] = "PAX_FRANQ"
        
        # 2. Si se llama TOTAL a secas, la pateamos a PAX_TOTAL 
        # (Para dejar libre el nombre 'TOTAL' para nuestra columna calculada)
        if col.upper() == "TOTAL" and "PAX_TOTAL" not in [c.upper() for c in columnas_actuales]:
                diccionario_renombrado[col] = "PAX_TOTAL"
            
        # Aplicamos los cambios si encontramos alguna coincidencia
        if diccionario_renombrado:
            df_actual = df_actual.rename(diccionario_renombrado)
            globals()[nombre_var] = df_actual
            print(f"Histórico {anio} modificado: {diccionario_renombrado}")
        else:
            print(f"Histórico {anio} ya estaba alineado.")

print("==================================================================")

REESTRUCTURACIÓN DE NOMENCLATURAS PARA EL DATASET
Histórico 2014 ya estaba alineado.
Histórico 2015 modificado: {'total': 'PAX_TOTAL'}
Histórico 2016 modificado: {'PAX_FREQ': 'PAX_FRANQ', 'TOTAL': 'PAX_TOTAL'}
Histórico 2017 modificado: {'PAX_FREQ': 'PAX_FRANQ', 'TOTAL': 'PAX_TOTAL'}
Histórico 2018 modificado: {'total': 'PAX_TOTAL'}
Histórico 2019 modificado: {'total': 'PAX_TOTAL'}
Histórico 2020 ya estaba alineado.
Histórico 2021 ya estaba alineado.


In [8]:
# Verificamos si el DataFrame de 2014 está en memoria
if "historico_2014_df" in globals():
    
    # Validamos si tiene la columna en singular
    if "PAX_PAGO" in historico_2014_df.columns:
        
        # Renombramos de forma estricta a plural
        historico_2014_df = historico_2014_df.rename({"PAX_PAGO": "PAX_PAGOS"})
        
        print("==================================================================")
        print("ALINEACIÓN DE VARIABLES DE CONTEO")
        print("==================================================================")
        print("Columna 'PAX_PAGO' renombrada exitosamente a 'PAX_PAGOS' en 2014.")
        print(f"• Columnas actuales del 2014: {historico_2014_df.columns}")
        print("-" * 66)

ALINEACIÓN DE VARIABLES DE CONTEO
Columna 'PAX_PAGO' renombrada exitosamente a 'PAX_PAGOS' en 2014.
• Columnas actuales del 2014: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
------------------------------------------------------------------


In [9]:
print("==================================================================")
print("ESTANDARIZACIÓN DE COLUMNAS A MAYÚSCULAS")
print("==================================================================")

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # Aplicamos el renombrado masivo usando una función lambda
        df_mayusculas = df_actual.rename(lambda col_name: col_name.upper())
        
        # Devolvemos el DataFrame homogeneizado al entorno global
        globals()[nombre_var] = df_mayusculas
        
        print(f"Histórico {anio} estandarizado:")
        print(f"   • Columnas finales: {df_mayusculas.columns}")
        print("-" * 66)

ESTANDARIZACIÓN DE COLUMNAS A MAYÚSCULAS
Histórico 2014 estandarizado:
   • Columnas finales: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
------------------------------------------------------------------
Histórico 2015 estandarizado:
   • Columnas finales: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
------------------------------------------------------------------
Histórico 2016 estandarizado:
   • Columnas finales: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
------------------------------------------------------------------
Histórico 2017 estandarizado:
   • Columnas finales: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
------------------------------------------------------------------
His

Se carga el resto de los dataset

In [10]:
lineas_subte_df = pl.read_csv('lineas-de-subte.csv', encoding='latin-1', null_values=null_values)
estaciones_accesibles_df = pl.read_csv('estaciones-accesibles.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_df = pl.read_csv('registro-historico-del-precio-del-boleto.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_excel_df = pl.read_excel('registro-historico-del-precio-del-boleto.xlsx')
viajes_anual_df = pl.read_csv('viajes_anual.csv', encoding='latin-1', null_values=null_values)

In [11]:
"""
# Esta es la anterior versión donde se cargaban todas las columnas

# Se cargan archivos en dataframes de polars
# Se agrega el encoding latin 1 para evitar errores

null_values = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]

lineas_subte_df = pl.read_csv('lineas-de-subte.csv', encoding='latin-1', null_values=null_values)

estaciones_accesibles_df = pl.read_csv('estaciones-accesibles.csv', encoding='latin-1', null_values=null_values)

historico_2014_df = pl.read_csv('historico_2014.csv', encoding='latin-1', null_values=null_values)
historico_2015_df = pl.read_csv('historico_2015.csv', encoding='latin-1', null_values=null_values)
historico_2016_df = pl.read_csv('historico_2016.csv', encoding='latin-1', null_values=null_values)
historico_2017_df = pl.read_csv('historico_2017.csv', encoding='latin-1', null_values=null_values)
historico_2018_df = pl.read_csv('historico_2018.csv', encoding='latin-1', null_values=null_values)
historico_2019_df = pl.read_csv('historico_2019.csv', encoding='latin-1', null_values=null_values)
historico_2020_df = pl.read_csv('historico_2020.csv', encoding='latin-1', null_values=null_values)
historico_2021_df = pl.read_csv('historico_2021.csv', encoding='latin-1', null_values=null_values, separator=';')
registro_historico_del_precio_del_boleto_df = pl.read_csv('registro-historico-del-precio-del-boleto.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_excel_df = pl.read_excel('registro-historico-del-precio-del-boleto.xlsx')
viajes_anual_df = pl.read_csv('viajes_anual.csv', encoding='latin-1', null_values=null_values)
"""

'\n# Esta es la anterior versión donde se cargaban todas las columnas\n\n# Se cargan archivos en dataframes de polars\n# Se agrega el encoding latin 1 para evitar errores\n\nnull_values = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]\n\nlineas_subte_df = pl.read_csv(\'lineas-de-subte.csv\', encoding=\'latin-1\', null_values=null_values)\n\nestaciones_accesibles_df = pl.read_csv(\'estaciones-accesibles.csv\', encoding=\'latin-1\', null_values=null_values)\n\nhistorico_2014_df = pl.read_csv(\'historico_2014.csv\', encoding=\'latin-1\', null_values=null_values)\nhistorico_2015_df = pl.read_csv(\'historico_2015.csv\', encoding=\'latin-1\', null_values=null_values)\nhistorico_2016_df = pl.read_csv(\'historico_2016.csv\', encoding=\'latin-1\', null_values=null_values)\nhistorico_2017_df = pl.read_csv(\'historico_2017.csv\', encoding=\'latin-1\', null_values=null_values)\nhistorico_2018_df = pl.read_csv(\'historico_2018.csv\', encoding=\'latin-1\', null_values=null_values)\nhistorico_201

### Análisis exploratorio de los dataframes

In [12]:
print("lineas_subte - Column names ", lineas_subte_df.columns)
display(lineas_subte_df.head())

print ('\n\n---------------------\n\n')

print("estaciones_accesibles - Column names:", estaciones_accesibles_df.columns)
display(estaciones_accesibles_df.head())

print ('\n\n---------------------\n\n')
print("historico_2014 - Column names:", historico_2014_df.columns)
display(historico_2014_df.head())

print ('\n\n---------------------\n\n')

print("historico_2015 - Column names:", historico_2015_df.columns)
display(historico_2015_df.head())

print ('\n\n---------------------\n\n')

print("historico_2016 - Column names:", historico_2016_df.columns)
display(historico_2016_df.head())

print ('\n\n---------------------\n\n')

print("historico_2017 - Column names:", historico_2017_df.columns)
display(historico_2017_df.head())

print ('\n\n---------------------\n\n')

print("historico_2018 - Column names:", historico_2018_df.columns)
display(historico_2018_df.head())

print ('\n\n---------------------\n\n')

print("historico_2019 - Column names:", historico_2019_df.columns)
display(historico_2019_df.head())

print ('\n\n---------------------\n\n')

print("historico_2020 - Column names:", historico_2020_df.columns)
display(historico_2020_df.head())

print ('\n\n---------------------\n\n')

print("historico_2021 - Column names:", historico_2021_df.columns)
display(historico_2021_df.head())

print ('\n\n---------------------\n\n')


lineas_subte - Column names  ['wkt', 'id', 'lineasub']


wkt,id,lineasub
str,i64,str
"""MULTILINESTRING ((-58.45212560…",1,"""LINEA D"""
"""MULTILINESTRING ((-58.45648913…",2,"""LINEA D"""
"""MULTILINESTRING ((-58.44466814…",3,"""LINEA D"""
"""MULTILINESTRING ((-58.43501353…",4,"""LINEA D"""
"""MULTILINESTRING ((-58.42571144…",5,"""LINEA D"""




---------------------


estaciones_accesibles - Column names: ['long', 'lat', 'linea', 'estacion', 'escaleras_mecanicas', 'ascensores']


long,lat,linea,estacion,escaleras_mecanicas,ascensores
f64,f64,str,str,i64,i64
-58.436429,-34.61828,"""A""","""ACOYTE""",2,2
-58.45671,-34.626667,"""A""","""CARABOBO""",3,3
-58.421816,-34.61177,"""A""","""CASTRO BARROS""",1,1
-58.392669,-34.609226,"""A""","""CONGRESO""",2,2
-58.382232,-34.6091,"""A""","""LIMA""",1,0




---------------------


historico_2014 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


...



---------------------


historico_2020 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
str,str,str,str,str,str,str,str,str,str
"""01/01/2020""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_Acoyte_N_Turn01""","""Acoyte""","""1""","""0""","""0""","""1"""
"""01/01/2020""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_Carabobo_E_Turn02""","""Carabobo""","""6""","""0""","""0""","""6"""
"""01/01/2020""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_CBarros_N_Turn03""","""Castro Barros""","""3""","""0""","""1""","""4"""
"""01/01/2020""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_CBarros_S_Turn02""","""Castro Barros""","""2""","""0""","""0""","""2"""
"""01/01/2020""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_Congreso_N_Turn03""","""Congreso""","""2""","""0""","""0""","""2"""




---------------------


historico_2021 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
str,str,str,str,str,str,str,str,str,str
"""1/1/2021""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_Flores_Este_Turn02""","""Flores""","""1""","""0""","""0""","""1"""
"""1/1/2021""","""08:00:00""","""08:15:00""","""LineaA""","""LineaA_SanPedrito_Oeste_Turn06""","""San Pedrito""","""0""","""0""","""2""","""2"""
"""1/1/2021""","""08:00:00""","""08:15:00""","""LineaB""","""LineaB_Alem_N_Turn01""","""Leandro N. Alem""","""1""","""0""","""0""","""1"""
"""1/1/2021""","""08:00:00""","""08:15:00""","""LineaB""","""LineaB_JMRosas_Oeste_Turn05""","""Rosas""","""1""","""0""","""0""","""1"""
"""1/1/2021""","""08:00:00""","""08:15:00""","""LineaB""","""LineaB_Malabia_N_Turn02""","""Malabia""","""1""","""0""","""0""","""1"""




---------------------




In [13]:
print("==================================================================")
print("INVESTIGACIÓN DE MUTACIONES DE FECHA")
print("==================================================================")

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        col_fecha = [c for c in df_actual.columns if c.upper() == "FECHA"][0]
        
        # Seteamos el formato teórico que asumimos antes
        if anio == 2020:
            fmt = "%m/%d/%Y"
        elif anio in [2016, 2017, 2021]:
            fmt = "%d/%m/%Y"
        else:
            fmt = "%Y-%m-%d"
            
        # Intentamos la conversión en un vector de prueba estricto
        # El .errors_on_null() nos va a dejar ver qué filas NO pudieron convertirse
        fechas_rotas = (
            df_actual
            .select(pl.col(col_fecha).unique()) # Nos quedamos solo con las strings únicas de fecha
            .with_columns(
                pl.col(col_fecha).str.to_date(format=fmt, strict=False).alias("intento_fecha")
            )
            .filter(pl.col("intento_fecha").is_null()) # Filtramos las que fallaron
            .select(col_fecha)
        )
        
        print(f"HISTÓRICO {anio} (Formato testeado: '{fmt}'):")
        print(f"  • Cantidad de formatos de texto diferentes encontrados: {fechas_rotas.height}")
        if fechas_rotas.height > 0:
            # Te muestra las primeras 5 variantes de texto que rompieron el molde
            print(f"  • Muestra de textos que fallaron: {fechas_rotas.head(5).to_series().to_list()}")
        print("-" * 66)

INVESTIGACIÓN DE MUTACIONES DE FECHA
HISTÓRICO 2014 (Formato testeado: '%Y-%m-%d'):
  • Cantidad de formatos de texto diferentes encontrados: 0
------------------------------------------------------------------
HISTÓRICO 2015 (Formato testeado: '%Y-%m-%d'):
  • Cantidad de formatos de texto diferentes encontrados: 0
------------------------------------------------------------------
HISTÓRICO 2016 (Formato testeado: '%d/%m/%Y'):
  • Cantidad de formatos de texto diferentes encontrados: 0
------------------------------------------------------------------
HISTÓRICO 2017 (Formato testeado: '%d/%m/%Y'):
  • Cantidad de formatos de texto diferentes encontrados: 0
------------------------------------------------------------------
HISTÓRICO 2018 (Formato testeado: '%Y-%m-%d'):
  • Cantidad de formatos de texto diferentes encontrados: 1
  • Muestra de textos que fallaron: [None]
------------------------------------------------------------------
HISTÓRICO 2019 (Formato testeado: '%Y-%m-%d'):
  •

Conversión de las columnas de fecha a tipo fecha

In [14]:
for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # 1. Identificamos columnas numéricas de pasajeros (ya sean str o i64/f64)
        columnas_pax_reales = [
            c for c in df_actual.columns 
            if "PAX_" in c.upper() or c.upper() == "TOTAL"
        ]
        col_fecha = [c for c in df_actual.columns if c.upper() == "FECHA"][0]
        
        # 2. Filtramos filas vacías de fecha
        df_actual = df_actual.filter(
            (pl.col(col_fecha).str.strip_chars() != "") & 
            (pl.col(col_fecha).is_not_null())
        )
        
        # 3. PROCESAMIENTO QUIRÚRGICO DE PASAJEROS
        operaciones = []
        for c in columnas_pax_reales:
            # Si vino como texto (por el infer_schema=0), limpiamos y casteamos
            if df_actual.schema[c] == pl.String:
                expr = pl.col(c).str.replace_all(" ", "").cast(pl.Float64, strict=False)
            else:
                # Si ya era número, la dejamos como está
                expr = pl.col(c)
                
            # EL SALVAVIDAS: Reemplazamos los nulos reales por 0
            expr = expr.fill_null(0)
            operaciones.append(expr)
            
        df_corregido = df_actual.with_columns(operaciones)
        
        # 4. Parseo Elástico de Fechas (Coalesce)
        # Solo si la fecha sigue siendo string
        if df_corregido.schema[col_fecha] == pl.String:
            intento_iso = pl.col(col_fecha).str.to_date(format="%Y-%m-%d", strict=False)
            intento_latino = pl.col(col_fecha).str.to_date(format="%d/%m/%Y", strict=False)
            intento_yanki = pl.col(col_fecha).str.to_date(format="%m/%d/%Y", strict=False)
            
            df_corregido = df_corregido.with_columns(
                pl.coalesce([intento_iso, intento_latino, intento_yanki]).alias(col_fecha)
            )
            
        globals()[nombre_var] = df_corregido
        
        print(f"Curación completada para el Histórico {anio}:")
        print(f"   • Columnas imputadas con 0 en caso de null: {columnas_pax_reales}")
        print(f"   • Nulos remanentes en FECHA: {df_corregido[col_fecha].is_null().sum()}")
        print("-" * 60)

Curación completada para el Histórico 2014:
   • Columnas imputadas con 0 en caso de null: ['PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
   • Nulos remanentes en FECHA: 0
------------------------------------------------------------
Curación completada para el Histórico 2015:
   • Columnas imputadas con 0 en caso de null: ['PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
   • Nulos remanentes en FECHA: 0
------------------------------------------------------------
Curación completada para el Histórico 2016:
   • Columnas imputadas con 0 en caso de null: ['PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
   • Nulos remanentes en FECHA: 0
------------------------------------------------------------
Curación completada para el Histórico 2017:
   • Columnas imputadas con 0 en caso de null: ['PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
   • Nulos remanentes en FECHA: 0
------------------------------------------------------------
Curación completada 

Conversión de las columnas tipo tiempo a tiempo

Conversión de las columnas tipo número a float

In [15]:
for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # 1. Buscamos las columnas de hora que existan en este año (en mayúsculas o minúsculas)
        cols_hora = [c for c in df_actual.columns if c.upper() in ["DESDE", "HASTA"]]
        
        if not cols_hora:
            continue
            
        # 2. Verificamos si al menos una todavía es String para procesar
        si_son_string = any(df_actual.schema[c] == pl.String for c in cols_hora)
        
        if si_son_string:
            # Diccionario temporal para aplicar las transformaciones
            transformaciones = {}
            
            for c in cols_hora:
                if df_actual.schema[c] == pl.String:
                    # Limpiamos espacios locos adelante y atrás
                    expr_limpia = pl.col(c).str.strip_chars()
                    
                    # Truco de elasticidad: Si la hora vino sin segundos (ej: "08:00"), 
                    # le pegamos el ":00" al final para que Polars no llore
                    expr_limpia = pl.when(expr_limpia.str.len_chars() == 5)\
                                    .then(expr_limpia + ":00")\
                                    .otherwise(expr_limpia)
                    
                    # Convertimos a Time (strict=False para que si hay una mugre insalvable ponga null y no explote)
                    transformaciones[c] = expr_limpia.str.to_time(format="%H:%M:%S", strict=False)
            
            df_corregido = df_actual.with_columns(**transformaciones)
            msg = f"Convertidas a pl.Time columnas: {cols_hora}"
        else:
            df_corregido = df_actual
            msg = "Ya eran tipo pl.Time (Omitido por seguridad)"
            
        globals()[nombre_var] = df_corregido
        
        print(f"Control de Horarios para el Histórico {anio}:")
        print(f"   • {msg}")
        print(f"   • Tipos actuales -> " + ", ".join([f"{c}: {df_corregido.schema[c]}" for c in cols_hora]))
        print("-" * 60)

Control de Horarios para el Histórico 2014:
   • Convertidas a pl.Time columnas: ['DESDE', 'HASTA']
   • Tipos actuales -> DESDE: Time, HASTA: Time
------------------------------------------------------------
Control de Horarios para el Histórico 2015:
   • Convertidas a pl.Time columnas: ['DESDE', 'HASTA']
   • Tipos actuales -> DESDE: Time, HASTA: Time
------------------------------------------------------------
Control de Horarios para el Histórico 2016:
   • Convertidas a pl.Time columnas: ['DESDE', 'HASTA']
   • Tipos actuales -> DESDE: Time, HASTA: Time
------------------------------------------------------------
Control de Horarios para el Histórico 2017:
   • Convertidas a pl.Time columnas: ['DESDE', 'HASTA']
   • Tipos actuales -> DESDE: Time, HASTA: Time
------------------------------------------------------------
Control de Horarios para el Histórico 2018:
   • Convertidas a pl.Time columnas: ['DESDE', 'HASTA']
   • Tipos actuales -> DESDE: Time, HASTA: Time
----------------

Rechequeo de tipos de columna

In [16]:
print("historico_2014 - Column names:", historico_2014_df.columns)
display(historico_2014_df.head())

print ('\n\n---------------------\n\n')

print("historico_2015 - Column names:", historico_2015_df.columns)
display(historico_2015_df.head())

print ('\n\n---------------------\n\n')

print("historico_2016 - Column names:", historico_2016_df.columns)
display(historico_2016_df.head())

print ('\n\n---------------------\n\n')

print("historico_2017 - Column names:", historico_2017_df.columns)
display(historico_2017_df.head())

print ('\n\n---------------------\n\n')

print("historico_2018 - Column names:", historico_2018_df.columns)
display(historico_2018_df.head())

print ('\n\n---------------------\n\n')

print("historico_2019 - Column names:", historico_2019_df.columns)
display(historico_2019_df.head())

print ('\n\n---------------------\n\n')

print("historico_2020 - Column names:", historico_2020_df.columns)
display(historico_2020_df.head())

print ('\n\n---------------------\n\n')

print("historico_2021 - Column names:", historico_2021_df.columns)
display(historico_2021_df.head())

print ('\n\n---------------------\n\n')

historico_2014 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
date,time,time,str,str,str,f64,f64,f64,f64
2014-04-09,09:15:00,09:29:00,"""B""","""LINEA_B_FLORIDA_E_TURN02""","""FLORIDA""",7.0,0.0,0.0,7.0
2014-04-09,09:15:00,09:29:00,"""B""","""LINEA_B_FLORIDA_E_TURN03""","""FLORIDA""",9.0,0.0,0.0,9.0
2014-04-09,09:15:00,09:29:00,"""B""","""LINEA_B_FLORIDA_O_TURN01""","""FLORIDA""",14.0,0.0,2.0,16.0
2014-04-09,09:15:00,09:29:00,"""B""","""LINEA_B_FLORIDA_O_TURN02""","""FLORIDA""",18.0,0.0,0.0,18.0
2014-04-09,09:15:00,09:29:00,"""B""","""LINEA_B_FLORIDA_O_TURN03""","""FLORIDA""",12.0,0.0,0.0,12.0




---------------------


historico_2015 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
date,time,time,str,str,str,f64,f64,f64,f64
2015-01-01,05:00:00,05:15:00,"""LINEA_H""","""LINEA_H_CASEROS_NORTE_TURN01""","""CASEROS""",0.0,0.0,0.0,0.0
2015-01-01,05:30:00,05:45:00,"""LINEA_A""","""LINEA_A_MISERERE_S_TURN03""","""PLAZA MISERERE""",0.0,0.0,0.0,0.0
2015-01-01,05:30:00,05:45:00,"""LINEA_D""","""LINEA_D_CATEDRAL_E_ASC01""","""CATEDRAL""",0.0,0.0,0.0,0.0
2015-01-01,05:30:00,05:45:00,"""LINEA_D""","""LINEA_D_CONGRESOTUC_O_TURN01""","""CONGRESO DE TUCUMAN""",0.0,0.0,0.0,0.0
2015-01-01,06:00:00,06:15:00,"""LINEA_C""","""LINEA_C_INDEPEN_TURN02""","""INDEPENDENCIA""",0.0,0.0,0.0,0.0




---------------------


historico_2016 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


...



---------------------


historico_2020 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
date,time,time,str,str,str,f64,f64,f64,f64
2020-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_Acoyte_N_Turn01""","""Acoyte""",1.0,0.0,0.0,1.0
2020-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_Carabobo_E_Turn02""","""Carabobo""",6.0,0.0,0.0,6.0
2020-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_CBarros_N_Turn03""","""Castro Barros""",3.0,0.0,1.0,4.0
2020-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_CBarros_S_Turn02""","""Castro Barros""",2.0,0.0,0.0,2.0
2020-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_Congreso_N_Turn03""","""Congreso""",2.0,0.0,0.0,2.0




---------------------


historico_2021 - Column names: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
date,time,time,str,str,str,f64,f64,f64,f64
2021-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_Flores_Este_Turn02""","""Flores""",1.0,0.0,0.0,1.0
2021-01-01,08:00:00,08:15:00,"""LineaA""","""LineaA_SanPedrito_Oeste_Turn06""","""San Pedrito""",0.0,0.0,2.0,2.0
2021-01-01,08:00:00,08:15:00,"""LineaB""","""LineaB_Alem_N_Turn01""","""Leandro N. Alem""",1.0,0.0,0.0,1.0
2021-01-01,08:00:00,08:15:00,"""LineaB""","""LineaB_JMRosas_Oeste_Turn05""","""Rosas""",1.0,0.0,0.0,1.0
2021-01-01,08:00:00,08:15:00,"""LineaB""","""LineaB_Malabia_N_Turn02""","""Malabia""",1.0,0.0,0.0,1.0




---------------------




### Exploración de estaciones accesibles sin escaleras mecánicas ni ascensores

In [17]:
# Filtramos usando .filter() y expresiones pl.col
resultado_df = estaciones_accesibles_df.filter(
    (pl.col("escaleras_mecanicas") == 0) & (pl.col("ascensores") == 0)
)

# Lo mostrás directamente (acordate que no hace falta display si es lo último)
resultado_df

long,lat,linea,estacion,escaleras_mecanicas,ascensores
f64,f64,str,str,i64,i64


### Observaciones

Se observa que no hay estaciones listadas como estaciones accesibles que posean 0 escaleras mecánicas y 0 ascensores, lo cual es correcto.

### Nulos

Al planificar el análisis predictivo con Prophet, no es necesario explorar o modificar los campos nulos en los campos numéricos. Se realiza el análisis en los campos literales para revisar que las etiquetas estén correctas y no cortar la continuidad de los datos.

In [18]:
# --- CONFIGURACIÓN DE VISUALIZACIÓN CORRECTA ---
# Forzamos a que no mutile los nombres de texto largos en la consola
pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(100)

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    # Verificamos si el DataFrame realmente existe en la memoria antes de procesarlo
    if nombre_var in globals():
        # Traemos el DataFrame usando su nombre en texto
        df_actual = globals()[nombre_var]
        
        # Ejecutamos el reporte de nulos y vacíos en formato vertical
        reporte = df_actual.select([
            pl.all().null_count().name.suffix("_nulos"),
            (pl.col(pl.String).str.strip_chars() == "").sum().name.suffix("_vacios")
        ]).transpose(include_header=True, header_name="Columna_Métrica", column_names=["Cantidad"])        
        
        # Imprimimos un encabezado claro para cada año
        print(f"=== REPORTE DE CALIDAD: HISTORICO {anio} ===")
        print(reporte)
        print("-" * 50, "\n")
    else:
        print(f"La variable {nombre_var} no está cargada en el entorno.\n")

=== REPORTE DE CALIDAD: HISTORICO 2014 ===
shape: (13, 2)
┌───────────────────────┬──────────┐
│ Columna_Métrica       ┆ Cantidad │
│ ---                   ┆ ---      │
│ str                   ┆ u32      │
╞═══════════════════════╪══════════╡
│ FECHA_nulos           ┆ 0        │
│ DESDE_nulos           ┆ 0        │
│ HASTA_nulos           ┆ 0        │
│ LINEA_nulos           ┆ 0        │
│ MOLINETE_nulos        ┆ 0        │
│ ESTACION_nulos        ┆ 0        │
│ PAX_PAGOS_nulos       ┆ 0        │
│ PAX_PASES_PAGOS_nulos ┆ 0        │
│ PAX_FRANQ_nulos       ┆ 0        │
│ PAX_TOTAL_nulos       ┆ 0        │
│ LINEA_vacios          ┆ 0        │
│ MOLINETE_vacios       ┆ 0        │
│ ESTACION_vacios       ┆ 0        │
└───────────────────────┴──────────┘
-------------------------------------------------- 

=== REPORTE DE CALIDAD: HISTORICO 2015 ===
shape: (13, 2)
┌───────────────────────┬──────────┐
│ Columna_Métrica       ┆ Cantidad │
│ ---                   ┆ ---      │
│ str            

In [19]:
# Miramos qué estaciones y molinetes tienen la línea en null
print(
    historico_2018_df
    .filter(pl.col("LINEA").is_null())
    .select(["ESTACION", "MOLINETE"])
    .unique()
)

shape: (1, 2)
┌──────────────────┬─────────────────┐
│ ESTACION         ┆ MOLINETE        │
│ ---              ┆ ---             │
│ str              ┆ str             │
╞══════════════════╪═════════════════╡
│ Taller Bonifacio ┆ Bonifacio_Tur02 │
└──────────────────┴─────────────────┘


In [20]:
with pl.Config(tbl_cols=-1, tbl_rows=100):
    
    # Filtramos y ordenamos cronológicamente para analizar el patrón semanal
    registros_bonifacio = (
        historico_2018_df
        .filter(pl.col("ESTACION") == "Taller Bonifacio")
        .sort("FECHA", "DESDE")
    )
    
    print(f"REGISTROS COMPLETOS - TALLER BONIFACIO (Total: {registros_bonifacio.height})")
    display(registros_bonifacio)

REGISTROS COMPLETOS - TALLER BONIFACIO (Total: 56)


FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
date,time,time,str,str,str,f64,f64,f64,f64
2018-04-03,08:30:00,08:45:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-03,10:00:00,10:15:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-03,10:45:00,11:00:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-04,10:45:00,11:00:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-05,07:30:00,07:45:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-05,09:15:00,09:30:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-06,09:00:00,09:15:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-06,09:15:00,09:30:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0
2018-04-06,10:30:00,10:45:00,null,"""Bonifacio_Tur02""","""Taller Bonifacio""",0.0,0.0,0.0,0.0


In [21]:
# Filtramos el DataFrame conservando todo, EXCEPTO lo que cumpla las dos condiciones de Bonifacio vacías
historico_2018_df = historico_2018_df.filter(
    ~((pl.col("ESTACION") == "Taller Bonifacio") & (pl.col("PAX_TOTAL") == 0.0))
)

### Conclusión de nulos

Se revisaron los nulos de los dataframes que los presentaban de forma consistente (en todos los campos)

En el caso del 2018, se presentaron 56 casos de campo "LINEA" nulo. Los registros corresponden a estación "Taller Bonifacio" y tienen todos sus números en cero. Se deduce que es entrada para empleados del subte y se eliminan las filas

### Duplicados

In [22]:
for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # 1. Duplicados Exactos: Filas que son un calco absoluto en TODAS sus columnas
        total_filas = df_actual.height
        filas_unicas_absolutas = df_actual.unique().height
        duplicados_exactos = total_filas - filas_unicas_absolutas
        
        # 2. Identificar dinámicamente las columnas clave de este año en particular
        # Buscamos cómo se escriben 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion' sin importar mayúsculas/minúsculas
        columnas_actuales = df_actual.columns
        columnas_clave = [
            col for col in columnas_actuales 
            if col.upper() in ["FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"]
        ]
        
        # 3. Duplicados por Clave: Mismo momento, mismo molinete, misma estación
        # (Aunque difieran en la columna de pasajeros por algún error de carga)
        filas_unicas_clave = df_actual.unique(subset=columnas_clave).height
        duplicados_clave = total_filas - filas_unicas_clave
        
        # Imprimimos el reporte formal para el informe de la materia
        print(f"=== REPORTE DE DUPLICADOS: HISTORICO {anio} ===")
        print(f"• Total de registros evaluados: {total_filas:,}")
        print(f"• Filas idénticas (100% duplicadas): {duplicados_exactos:,}")
        print(f"• Duplicados por clave espacio-temporal: {duplicados_clave:,}")
        
        if duplicados_exactos == 0 and duplicados_clave == 0:
            print("Conclusión: El dataset no presenta anomalías de duplicación en esta etapa.")
        else:
            print("Conclusión: Se detectaron registros duplicados a tratar en la fase de corrección.")
            
        print("-" * 60, "\n")

=== REPORTE DE DUPLICADOS: HISTORICO 2014 ===
• Total de registros evaluados: 10,857,244
• Filas idénticas (100% duplicadas): 0
• Duplicados por clave espacio-temporal: 0
Conclusión: El dataset no presenta anomalías de duplicación en esta etapa.
------------------------------------------------------------ 

=== REPORTE DE DUPLICADOS: HISTORICO 2015 ===
• Total de registros evaluados: 10,958,582
• Filas idénticas (100% duplicadas): 0
• Duplicados por clave espacio-temporal: 0
Conclusión: El dataset no presenta anomalías de duplicación en esta etapa.
------------------------------------------------------------ 

=== REPORTE DE DUPLICADOS: HISTORICO 2016 ===
• Total de registros evaluados: 11,542,322
• Filas idénticas (100% duplicadas): 0
• Duplicados por clave espacio-temporal: 0
Conclusión: El dataset no presenta anomalías de duplicación en esta etapa.
------------------------------------------------------------ 

=== REPORTE DE DUPLICADOS: HISTORICO 2017 ===
• Total de registros evalua

In [23]:
# Lista de los dos años con problemas
anios_a_consolidar = [2020, 2021]

for anio in anios_a_consolidar:
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_cronico = globals()[nombre_var]
        
        # Paso 1: Eliminar las filas 100% idénticas (las de 14k y 15k)
        df_depurado = df_cronico.unique()
        
        # Paso 2: Agrupar por clave espacio-temporal para consolidar los tipos de pasajes
        # Agrupamos por la infraestructura y el tiempo, y sumamos las métricas numéricas
        df_consolidado = (
            df_depurado
            .group_by(["FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"])
            .agg([
                pl.col("PAX_PAGOS").sum(),
                pl.col("PAX_PASES_PAGOS").sum(),
                pl.col("PAX_FRANQ").sum(),
                pl.col("PAX_TOTAL").sum()
            ])
        )
        
        # Devolvemos la matriz sana al entorno global
        globals()[nombre_var] = df_consolidado
        
        print(f"Consolidación terminada para el Histórico {anio}:")
        print(f"   • Registros originales: {df_cronico.shape[0]:,}")
        print(f"   • Registros post-curación: {df_consolidado.shape[0]:,}")
        print(f"   • Filas reducidas (Limpieza + Colapso): {df_cronico.shape[0] - df_consolidado.shape[0]:,}\n")
        print("-" * 70)

Consolidación terminada para el Histórico 2020:
   • Registros originales: 5,779,170
   • Registros post-curación: 5,460,441
   • Filas reducidas (Limpieza + Colapso): 318,729

----------------------------------------------------------------------
Consolidación terminada para el Histórico 2021:
   • Registros originales: 8,071,680
   • Registros post-curación: 7,792,749
   • Filas reducidas (Limpieza + Colapso): 278,931

----------------------------------------------------------------------


En los reportes de 2020 y 2021 se encuentran filas idénticas (duplicadas). Se genera un análisis de esas filas para su corrección.

También se encuentran filas duplicadas por clave espacio-temporal ("FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"), se genera un agrupamiento de esas fila para unificar los registros.

"Durante la auditoría de unicidad en los periodos de pandemia y post-pandemia (2020-2021), se identificó un cambio estructural en el paradigma de registro de la fuente. A diferencia del histórico previo, el sistema atomizó los registros duplicando las claves espacio-temporales (FECHA, FRANJA_HORARIA, MOLINETE) para asentar de forma segregada los vectores de transacciones según la tipología del usuario (PAX_PAGOS, PAX_PASES, PAX_FRANQ). Para homogeneizar la granularidad con el resto del histórico sin incurrir en pérdida de información, se procedió a una remoción inicial de duplicados sintácticos absolutos, seguida de una operación de agregación distributiva (group_by con acumulación por suma), consolidando la unidad mínima muestral."

### Outliers

In [24]:
print("==================================================================")
print("AUDITORÍA DE OUTLIERS NUMÉRICOS: PASAJEROS TOTALES")
print("==================================================================")

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # Identificamos la columna del total de pasajeros de forma dinámica
        col_total = [c for c in df_actual.columns if "PAX_TOTAL" in c.upper()][0]
        
        # Calculamos estadísticas clave sobre esa columna
        stats = df_actual.select([
            pl.col(col_total).min().alias("minimo"),
            pl.col(col_total).max().alias("maximo"),
            pl.col(col_total).mean().alias("media"),
            pl.col(col_total).std().alias("desvio_estandar"),
            # Contamos si hay algún negativo colgado
            (pl.col(col_total) < 0).sum().alias("cant_negativos")
        ])
        
        print(f"HISTÓRICO {anio} (Columna: '{col_total}'):")
        print(f"  • Valor Mínimo: {stats['minimo'][0]}")
        print(f"  • Valor Máximo: {stats['maximo'][0]:,}")
        print(f"  • Media / Promedio: {stats['media'][0]:.2f}")
        print(f"  • Desvío Estándar: {stats['desvio_estandar'][0]:.2f}")
        print(f"  • Registros Negativos: {stats['cant_negativos'][0]}")
        
        # Alerta metodológica si el máximo es absurdamente gigante para 15 minutos
        if stats['maximo'][0] > 15000:
            print(f"ALERTA: El valor máximo parece físicamente imposible para un molinete en 15 min.")
        if stats['cant_negativos'][0] > 0:
            print(f"ALERTA: Hay valores negativos que romperían el modelo.")
            
        print("-" * 66)

AUDITORÍA DE OUTLIERS NUMÉRICOS: PASAJEROS TOTALES
HISTÓRICO 2014 (Columna: 'PAX_TOTAL'):
  • Valor Mínimo: 0.0
  • Valor Máximo: 14,538.0
  • Media / Promedio: 23.50
  • Desvío Estándar: 27.73
  • Registros Negativos: 0
------------------------------------------------------------------
HISTÓRICO 2015 (Columna: 'PAX_TOTAL'):
  • Valor Mínimo: 0.0
  • Valor Máximo: 423.0
  • Media / Promedio: 25.78
  • Desvío Estándar: 28.99
  • Registros Negativos: 0
------------------------------------------------------------------
HISTÓRICO 2016 (Columna: 'PAX_TOTAL'):
  • Valor Mínimo: 0.0
  • Valor Máximo: 362.0
  • Media / Promedio: 27.24
  • Desvío Estándar: 30.05
  • Registros Negativos: 0
------------------------------------------------------------------
HISTÓRICO 2017 (Columna: 'PAX_TOTAL'):
  • Valor Mínimo: 0.0
  • Valor Máximo: 399.0
  • Media / Promedio: 27.53
  • Desvío Estándar: 30.12
  • Registros Negativos: 0
------------------------------------------------------------------
HISTÓRICO 

In [26]:
print("==================================================================")
print("🔍 AUDITORÍA DE OUTLIERS TEMPORALES (FECHAS Y FRANJAS)")
print("==================================================================")

for anio in range(2014, 2022):
    nombre_var = f"historico_{anio}_df"
    
    if nombre_var in globals():
        df_actual = globals()[nombre_var]
        
        # Mapeamos dinámicamente las columnas temporales
        col_fecha = [c for c in df_actual.columns if c.upper() == "FECHA"][0]
        col_desde = [c for c in df_actual.columns if c.upper() == "DESDE"][0]
        col_hasta = [c for c in df_actual.columns if c.upper() == "HASTA"][0]
        
        # CORRECCIÓN 1: Filtramos solo nulos reales porque FECHA ya es pl.Date
        df_limpio_temp = df_actual.filter(pl.col(col_fecha).is_not_null())
        
        # CORRECCIÓN 2: Usamos el método de fechas .dt.year() en vez de cortar texto con .str.slice()
        fechas_fuera_de_anio = df_limpio_temp.filter(
            pl.col(col_fecha).dt.year() != anio
        )
        
        # 2. Validamos que el formato de hora sea correcto (como DESDE ya es pl.Time, 
        # cualquier registro deformado se habría convertido en null antes. Contamos los nulls en DESDE)
        horas_desde_raras = df_limpio_temp.filter(pl.col(col_desde).is_null())
        
        print(f"📅 TEMPORAL {anio}:")
        print(f"  • Registros con fechas que NO corresponden al año {anio}: {fechas_fuera_de_anio.height}")
        print(f"  • Registros con formatos de hora rotos/nulos: {horas_desde_raras.height}")
        
        if fechas_fuera_de_anio.height > 0:
            print(f"  ⚠️ ALERTA: Hay fechas mezcladas de otros años en este archivo.")
            print("    Años detectados:", fechas_fuera_de_anio.select(pl.col(col_fecha).dt.year()).unique().to_series().to_list())
            
        print("-" * 66)

🔍 AUDITORÍA DE OUTLIERS TEMPORALES (FECHAS Y FRANJAS)
📅 TEMPORAL 2014:
  • Registros con fechas que NO corresponden al año 2014: 0
  • Registros con formatos de hora rotos/nulos: 0
------------------------------------------------------------------
📅 TEMPORAL 2015:
  • Registros con fechas que NO corresponden al año 2015: 0
  • Registros con formatos de hora rotos/nulos: 0
------------------------------------------------------------------
📅 TEMPORAL 2016:
  • Registros con fechas que NO corresponden al año 2016: 0
  • Registros con formatos de hora rotos/nulos: 0
------------------------------------------------------------------
📅 TEMPORAL 2017:
  • Registros con fechas que NO corresponden al año 2017: 0
  • Registros con formatos de hora rotos/nulos: 0
------------------------------------------------------------------
📅 TEMPORAL 2018:
  • Registros con fechas que NO corresponden al año 2018: 0
  • Registros con formatos de hora rotos/nulos: 0
-------------------------------------------

https://github.com/AleLoredo/UGR-metodologia/blob/eze/UGR_Metodologia_TP1_subte.ipynb